In [ ]:
"""
============================================================
Simple PDF RAG Chatbot
------------------------------------------------------------
This script:

1. Reads a PDF
2. Splits it into chunks
3. Creates embeddings
4. Stores them in ChromaDB
5. Retrieves relevant chunks
6. Answers questions using GPT

Compatible with VS Code.
============================================================
"""

import os
import uuid
import textwrap

from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
import chromadb

# ============================================================
# LOAD ENVIRONMENT VARIABLES
# ============================================================

# Load variables from .env
load_dotenv()

# Read API key
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY not found in .env file")

# ============================================================
# OPENAI CLIENT
# ============================================================

client = OpenAI(api_key=OPENAI_API_KEY)

# ============================================================
# CHROMADB SETUP
# ============================================================

# Persist data locally
chroma_client = chromadb.PersistentClient(path="./chroma_db")

collection = chroma_client.get_or_create_collection(
    name="simple_rag"
)

# ============================================================
# LOAD PDF
# ============================================================

def load_pdf(pdf_path):
    """
    Reads a PDF and returns all text as a string.
    """

    reader = PdfReader(pdf_path)

    text = ""

    for page in reader.pages:

        extracted_text = page.extract_text()

        if extracted_text:
            text += extracted_text + "\n"

    return text


# ============================================================
# TEXT CHUNKING
# ============================================================

def chunk_text(text, chunk_size=800, overlap=200):
    """
    Splits text into overlapping chunks.

    Example:

    Chunk 1:
    0 -------- 800

    Chunk 2:
           600 -------- 1400

    Overlap preserves context between chunks.
    """

    chunks = []

    start = 0

    while start < len(text):

        end = start + chunk_size

        chunk = text[start:end]

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


# ============================================================
# CREATE EMBEDDING
# ============================================================

def get_embedding(text):
    """
    Generates an embedding vector for the given text.
    """

    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )

    return response.data[0].embedding


# ============================================================
# STORE CHUNKS IN CHROMADB
# ============================================================

def add_chunks_to_chroma(chunks):
    """
    Stores document chunks and embeddings inside ChromaDB.
    """

    print("\nCreating embeddings...")

    for chunk in chunks:

        embedding = get_embedding(chunk)

        collection.add(
            ids=[str(uuid.uuid4())],
            documents=[chunk],
            embeddings=[embedding]
        )

    print(f"\nStored {len(chunks)} chunks successfully.")


# ============================================================
# RETRIEVE RELEVANT CHUNKS
# ============================================================

def retrieve(query, top_k=3):
    """
    Finds the most relevant document chunks.
    """

    query_embedding = get_embedding(query)

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k
    )

    return results["documents"][0]


# ============================================================
# ASK QUESTION
# ============================================================

def ask(query):
    """
    Retrieves relevant chunks and asks GPT to answer
    using ONLY the retrieved context.
    """

    docs = retrieve(query)

    context = "\n\n".join(docs)

    prompt = f"""
You are a helpful RAG assistant.

Answer ONLY using the provided context.

If the answer is not present, respond exactly:

"I could not find the answer in the document."

Context:
{context}

Question:
{query}
"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0.2,
        messages=[
            {
                "role": "system",
                "content": "Answer only using retrieved documents."
            },
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response.choices[0].message.content


# ============================================================
# BUILD RAG DATABASE
# ============================================================

def build_rag(pdf_path):
    """
    Reads PDF, chunks it, and stores embeddings.
    """

    print("\nLoading PDF...")

    text = load_pdf(pdf_path)

    print("PDF Loaded.")

    print("\nChunking document...")

    chunks = chunk_text(
        text,
        chunk_size=800,
        overlap=200
    )

    print(f"Created {len(chunks)} chunks.")

    # Prevent duplicate insertion if collection already has data
    if collection.count() == 0:
        add_chunks_to_chroma(chunks)
    else:
        print("ChromaDB already contains data. Skipping embedding generation.")


# ============================================================
# CHAT LOOP
# ============================================================

def chatbot():
    """
    Interactive chatbot loop.
    """

    print("\n===========================================")
    print("        PDF RAG CHATBOT")
    print("===========================================")

    print("Type 'exit' to quit.\n")

    while True:

        query = input("You: ")

        if query.lower() == "exit":

            print("\nBot: Goodbye!")

            break

        answer = ask(query)

        print("\nBot:\n")

        print(textwrap.fill(answer, width=100))

        print("\n" + "=" * 100 + "\n")


# ============================================================
# MAIN FUNCTION
# ============================================================

def main():
    """
    Program entry point.
    """

    pdf_path = input("Enter PDF path: ").strip()

    if not os.path.exists(pdf_path):

        print("\nPDF not found.")

        return

    build_rag(pdf_path)

    chatbot()


# ============================================================
# RUN PROGRAM
# ============================================================

if __name__ == "__main__":
    main()